<a href="https://colab.research.google.com/github/haidermalik68/Ai-image-checker/blob/main/03_evaluation_and_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##3.0 Imports and Drive

In [19]:
import io
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from google.colab import drive

drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)    # expect cuda

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using: cpu


In [12]:
## bring back images and tables
!cd /content && unzip -q /content/drive/MyDrive/ai_image_checker/data.zip
!cp /content/drive/MyDrive/ai_image_checker/*.csv /content/

test = pd.read_csv("/content/test.csv")
print(len(test))    # expect 12600

replace data/sd21/1040.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 12600


### 3.0-c. Transforms and dataset class

In [13]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class ImageDataset(Dataset):
    def __init__(self, table, transform):
        self.table = table.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, i):
        row = self.table.iloc[i]
        image = Image.open(row["file"]).convert("RGB")
        return self.transform(image), int(row["label"])

## 3.0-D.Rebuild the model and load your best weights

In [14]:
save_folder = "/content/drive/MyDrive/ai_image_checker"

model = models.resnet18(weights=None)               # no ImageNet download, our weights come from the file
model.fc = nn.Linear(model.fc.in_features, 2)       # same shape as when we trained

model.load_state_dict(torch.load(save_folder + "/best_model.pt", map_location=device))
model = model.to(device)
model.eval()                                        # testing mode
print("model loaded")

model loaded


##3.1 Robustness: do the scores survive real-world damage?
* Photos shared on WhatsApp or social media are compressed and resized, so a detector that only works on clean files isn't useful.
* We re-test the same 12,600 test images after two kinds of damage: stronger JPEG compression (quality 50) and shrinking to 128×128 and enlarging back.
* A big drop means the model relied on fragile fine details.
This also helps check the shrink-amount shortcut idea from the README.
* Nothing is trained or changed here. We only measure.

## 3.1-A: Damage Functions

define two functions that take an image and return a damaged copy

In [31]:
def degraded_jpeg(image,quality):
  buffer  = io.BytesIO()
  image.save(buffer,format="JPEG",quality=quality)
  buffer.seek(0)
  return Image.open(buffer)

def degraded_resize(image,size):
  small = image.resize((size,size))
  return small.resize((256,256))

## 3.1-B. Transforms with damage

In [32]:
## two transforms that damage the image first and then apply te normal test steps
jpeg50_transform = transforms.Compose([
    transforms.Lambda(lambda img: degraded_jpeg(img,50)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

shrink128_transform = transforms.Compose([
    transforms.Lambda(lambda img: degraded_resize(img,128)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])




In [33]:
def score_per_generator(transform, table):
    data = ImageDataset(table, transform)
    loader = DataLoader(data, batch_size=64, shuffle=False, num_workers=2)

    preds = []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            preds.extend(outputs.argmax(1).cpu().tolist())

    result = table.copy().reset_index(drop=True)
    result["correct"] = (pd.Series(preds) == result["label"])
    return result.groupby("generator")["correct"].mean().round(3)

In [34]:
##take a small sample of test dataset
small_test = test.groupby("generator").sample(n=200,random_state=0)
print(small_test["generator"].value_counts())

generator
dalle3        200
midjourney    200
real          200
sd21          200
sd3           200
sdxl          200
Name: count, dtype: int64


In [35]:
##3.1-D.  Run all three and compare
## scores the original,shrink128,jpeg50, making a table and compare them

comparison = pd.DataFrame({
    "original": score_per_generator(test_transform,small_test),
    "shrink128": score_per_generator(shrink128_transform,small_test),
    "jpeg50": score_per_generator(jpeg50_transform,small_test),
})
print(comparison)
comparison.to_csv(save_folder + "/robustness_results.csv")

            original  shrink128  jpeg50
generator                              
dalle3         0.045      0.310   0.020
midjourney     0.350      0.335   0.130
real           0.940      0.810   0.970
sd21           0.845      0.940   0.625
sd3            0.180      0.335   0.115
sdxl           0.955      0.955   0.815
